In [6]:
import os
from google.colab import drive

# 1. Buat folder laporan lokal di Colab (agar anggota tim lain tetap bisa Run)
REPORTS_DIR = '/content/reports'
os.makedirs(REPORTS_DIR, exist_ok=True)
print(f"Folder laporan lokal siap di: {REPORTS_DIR}")

# 2. Hubungkan Google Drive untuk backup permanen otomatis
USE_GOOGLE_DRIVE_BACKUP = True

try:
    drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = '/content/drive/MyDrive/IsyaratQuest_ML/reports'
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print(f"Backup Google Drive aktif di: {DRIVE_BACKUP_DIR}")
except Exception as e:
    print("Koneksi Google Drive dilewati, menggunakan penyimpanan lokal.")
    USE_GOOGLE_DRIVE_BACKUP = False

Folder laporan lokal siap di: /content/reports
Mounted at /content/drive
Backup Google Drive aktif di: /content/drive/MyDrive/IsyaratQuest_ML/reports


In [8]:
import os
import subprocess
import kagglehub

# 1. Kandidat 1: Kaggle (~312 Gambar - Rujukan UMN)
path_kandidat_1 = kagglehub.dataset_download("achmadnoer/alfabet-bisindo")
print("Lokasi Kandidat 1 (UMN):", path_kandidat_1)

# 2. Kandidat 2: Kaggle (~11.500 Gambar - Rujukan Unimal)
path_kandidat_2 = kagglehub.dataset_download("agungmrf/indonesian-sign-language-bisindo")
print("Lokasi Kandidat 2 (Unimal):", path_kandidat_2)

# 3. Kandidat 3: Mendeley Data (796 MB - Otomatis Drive / Download Publik)
path_kandidat_3 = '/content/dataset_mendeley'
zip_mendeley_local = '/content/mendeley_bisindo.zip'
os.makedirs(path_kandidat_3, exist_ok=True)

# Cek terlebih dahulu apakah file ZIP ada di Google Drive (jika Drive di-mount)
found_zips = []
if os.path.exists('/content/drive/MyDrive'):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if f.lower().endswith('.zip'):
                full_path = os.path.join(root, f)
                size_mb = os.path.getsize(full_path) / (1024 * 1024)
                found_zips.append((full_path, size_mb))

mendeley_candidates = [
    p for p, s in found_zips
    if any(k in os.path.basename(p).lower() for k in ['mendeley', 'bisindo', '4xnkvr']) or s > 500
]

if mendeley_candidates:
    zip_mendeley = mendeley_candidates[0]
    print(f"\nFile ditemukan di Google Drive: {zip_mendeley}")
    print("Sedang mengekstrak dataset Mendeley dari Drive (harap tunggu 1-2 menit)...")
    subprocess.run(["unzip", "-q", "-o", zip_mendeley, "-d", path_kandidat_3], check=True)
    print("Berhasil! Lokasi Kandidat 3 (Mendeley):", path_kandidat_3)
else:
    print("\nFile tidak ditemukan di Drive (atau dijalankan oleh anggota tim lain).")
    print("Mengunduh otomatis dari server publik Mendeley Data (796 MB, harap tunggu 1-3 menit)...")
    mendeley_url = "https://data.mendeley.com/public-files/datasets/4xnkvr88tk/files/archive.zip"
    subprocess.run(["wget", "-q", "--show-progress", "-O", zip_mendeley_local, mendeley_url], check=True)
    print("Sedang mengekstrak dataset Mendeley...")
    subprocess.run(["unzip", "-q", "-o", zip_mendeley_local, "-d", path_kandidat_3], check=True)
    print("Berhasil! Lokasi Kandidat 3 (Mendeley):", path_kandidat_3)

Using Colab cache for faster access to the 'alfabet-bisindo' dataset.
Lokasi Kandidat 1 (UMN): /kaggle/input/alfabet-bisindo
Using Colab cache for faster access to the 'indonesian-sign-language-bisindo' dataset.
Lokasi Kandidat 2 (Unimal): /kaggle/input/indonesian-sign-language-bisindo

File ditemukan di Google Drive: /content/drive/MyDrive/IsyaratQuest_ML/mendeley_bisindo.zip.zip
Sedang mengekstrak dataset Mendeley dari Drive (harap tunggu 1-2 menit)...
Berhasil! Lokasi Kandidat 3 (Mendeley): /content/dataset_mendeley


In [3]:
!pip install -q mediapipe opencv-python pandas

import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Unduh model Hand Landmarker jika belum tersedia
model_path = '/content/hand_landmarker.task'
if not os.path.exists(model_path):
    print("Mengunduh model Hand Landmarker...")
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    urllib.request.urlretrieve(url, model_path)

# Konfigurasi detektor untuk mendukung hingga 2 tangan
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.5
)
detector = vision.HandLandmarker.create_from_options(options)
print("MediaPipe Hand Landmarker (2 tangan) siap digunakan!")

Mengunduh model Hand Landmarker...
MediaPipe Hand Landmarker (2 tangan) siap digunakan!


In [9]:
import cv2
import pandas as pd

def collect_class_images(base_path):
    """Mengumpulkan seluruh file gambar berdasarkan nama folder kelas (A-Z)."""
    valid_ext = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    class_images = {}
    for root, dirs, files in os.walk(base_path):
        folder_name = os.path.basename(root).strip().upper()
        if len(folder_name) == 1 and folder_name.isalpha():
            imgs = [
                os.path.join(root, f) for f in files
                if f.lower().endswith(valid_ext)
            ]
            if imgs:
                class_images.setdefault(folder_name, []).extend(imgs)
    return class_images

def audit_dataset(dataset_name, raw_path, save_csv_name):
    print(f"\n==========================================")
    print(f"Memulai Audit: {dataset_name}")
    print(f"==========================================")

    class_map = collect_class_images(raw_path)
    if not class_map:
        print(f"Tidak ditemukan folder kelas A-Z di dalam: {raw_path}")
        return pd.DataFrame()

    audit_results = []

    for label in sorted(class_map.keys()):
        image_paths = class_map[label]
        total_images = len(image_paths)

        success_count = 0
        failed_count = 0
        one_hand_count = 0
        two_hands_count = 0

        for img_path in image_paths:
            img_bgr = cv2.imread(img_path)
            if img_bgr is None:
                failed_count += 1
                continue

            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

            result = detector.detect(mp_image)
            num_detected = len(result.hand_landmarks) if result.hand_landmarks else 0

            if num_detected > 0:
                success_count += 1
                if num_detected == 1:
                    one_hand_count += 1
                else:
                    two_hands_count += 1
            else:
                failed_count += 1

        failure_rate = round((failed_count / total_images) * 100, 2) if total_images > 0 else 0.0

        audit_results.append({
            'dataset': dataset_name,
            'class_label': label,
            'total_images': total_images,
            'success_images': success_count,
            'one_hand_detected': one_hand_count,
            'two_hands_detected': two_hands_count,
            'failed_images': failed_count,
            'failure_rate_pct': failure_rate
        })
        print(f"Kelas {label}: {total_images} sampel | Berhasil: {success_count} | Gagal: {failure_rate}%")

    df_audit = pd.DataFrame(audit_results)

    # Simpan ke folder lokal Colab
    local_csv = os.path.join(REPORTS_DIR, save_csv_name)
    df_audit.to_csv(local_csv, index=False)
    print(f"--> Tersimpan di Colab: {local_csv}")

    # Simpan otomatis ke Google Drive
    if USE_GOOGLE_DRIVE_BACKUP:
        drive_csv = os.path.join(DRIVE_BACKUP_DIR, save_csv_name)
        df_audit.to_csv(drive_csv, index=False)
        print(f"--> Backup tersimpan di Google Drive: {drive_csv}")

    return df_audit

In [10]:
# 1. Jalankan audit untuk ketiga kandidat dataset
df_kandidat1 = audit_dataset("Kandidat_1_Kaggle_UMN", path_kandidat_1, "audit_kandidat_1_umn.csv")
df_kandidat2 = audit_dataset("Kandidat_2_Kaggle_Unimal", path_kandidat_2, "audit_kandidat_2_unimal.csv")
df_kandidat3 = audit_dataset("Kandidat_3_Mendeley", path_kandidat_3, "audit_kandidat_3_mendeley.csv")

# 2. Fungsi perangkum untuk membandingkan kelayakan antar dataset
def summarize_dataset(df, name):
    if df.empty:
        return {
            'Nama Dataset': name,
            'Jumlah Kelas': 0,
            'Total Gambar': 0,
            'Total Berhasil': 0,
            'Terdeteksi 1 Tangan': 0,
            'Terdeteksi 2 Tangan': 0,
            'Total Gagal': 0,
            'Failure Rate (%)': 0.0,
            'Kelas Gagal >50%': 0
        }
    total_img = int(df['total_images'].sum())
    total_success = int(df['success_images'].sum())
    total_fail = int(df['failed_images'].sum())
    overall_fail_rate = round((total_fail / total_img) * 100, 2) if total_img > 0 else 0.0
    return {
        'Nama Dataset': name,
        'Jumlah Kelas': len(df),
        'Total Gambar': total_img,
        'Total Berhasil': total_success,
        'Terdeteksi 1 Tangan': int(df['one_hand_detected'].sum()),
        'Terdeteksi 2 Tangan': int(df['two_hands_detected'].sum()),
        'Total Gagal': total_fail,
        'Failure Rate (%)': overall_fail_rate,
        'Kelas Gagal >50%': int((df['failure_rate_pct'] > 50).sum())
    }

# 3. Cetak dan simpan tabel komparasi
df_comparison = pd.DataFrame([
    summarize_dataset(df_kandidat1, "Kandidat 1 (Kaggle ~312 - UMN)"),
    summarize_dataset(df_kandidat2, "Kandidat 2 (Kaggle ~11.500 - Unimal)"),
    summarize_dataset(df_kandidat3, "Kandidat 3 (Mendeley 796 MB)")
])

print("\n================ HASIL KOMPARASI 3 DATASET ================")
print(df_comparison.to_string(index=False))

comp_local = os.path.join(REPORTS_DIR, "audit_dataset_comparison.csv")
df_comparison.to_csv(comp_local, index=False)

if USE_GOOGLE_DRIVE_BACKUP:
    comp_drive = os.path.join(DRIVE_BACKUP_DIR, "audit_dataset_comparison.csv")
    df_comparison.to_csv(comp_drive, index=False)
    print(f"\nTabel komparasi permanen tersimpan di: {comp_drive}")


Memulai Audit: Kandidat_1_Kaggle_UMN
Kelas A: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas B: 12 sampel | Berhasil: 8 | Gagal: 33.33%
Kelas C: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas D: 12 sampel | Berhasil: 11 | Gagal: 8.33%
Kelas E: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas F: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas G: 12 sampel | Berhasil: 10 | Gagal: 16.67%
Kelas H: 12 sampel | Berhasil: 11 | Gagal: 8.33%
Kelas I: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas J: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas K: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas L: 12 sampel | Berhasil: 10 | Gagal: 16.67%
Kelas M: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas N: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas O: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas P: 12 sampel | Berhasil: 7 | Gagal: 41.67%
Kelas Q: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas R: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas S: 12 sampel | Berhasil: 12 | Gagal: 0.0%
Kelas T: 12 sampel | Berhasil: 10 | Gagal: